# Distance-Aware Hierarchical GNN for Electron Density Prediction

## Why Distance is the Right Inductive Bias

Electron density is governed by **Coulomb physics** — everything decays with distance.
- Density around an atom falls off as `exp(-α·r)` from the nucleus
- Inter-atomic influence decays as `1/r` (electrostatics) or `exp(-r)` (exchange-correlation)
- KNN by value is arbitrary. KNN by index is a file artifact. **Physical 3D distance is real.**

## What Changed from the Previous Hierarchical Notebook

| Component | Old (Hierarchical) | New (Distance-Aware) |
|---|---|---|
| Intra-patch edges | None (MLP only) | Distance-cutoff graph within each patch |
| Distance encoding | Raw `(x,y,z)` offset | **RBF (Radial Basis Functions)** — 16 Gaussian kernels |
| Patch pooling | `global_add_pool` (equal weights) | **Distance-weighted pooling** `exp(-d/σ)` |
| Inter-atom edges | No edge features | **Inter-atomic distances as edge_attr** to GATv2 |
| Decoder input | Atom vec + raw offset | Atom vec + **RBF(d_to_atom)** |
| Normalization | Per-snapshot (inconsistent) | **Global** mean/std over training set |
| Training | 5 epochs, no validation | Full loop: train/val, MAE, R², model saving |

In [ ]:
# ── 1. Install & Imports ──────────────────────────────────────────────────────
!pip -q install numpy pandas matplotlib seaborn scikit-learn torch-geometric scipy

import glob
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.spatial.distance import cdist
from torch_geometric.nn import GATv2Conv
from torch.utils.data import DataLoader, random_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device}')

np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# ── 2. Data Path Resolution ───────────────────────────────────────────────────
def resolve_raw_data_dir():
    candidates = [
        Path('../data/raw'),
        Path('./data/raw'),
        Path('/content/drive/MyDrive/Predicting-Electron-Interactions-as-Evolving-Graphs/data/raw'),
        Path('/home/varun/Desktop/someProj/GNNs/data/raw'),
    ]
    for p in candidates:
        if (p / 'ammonia_x').exists():
            return p.resolve()
    return Path('../data/raw')

DATA_DIR = resolve_raw_data_dir() / 'ammonia_x'
print(f'Data directory: {DATA_DIR}')

In [ ]:
# ── 3. Geometry Parsing ───────────────────────────────────────────────────────
def parse_xyz(filepath):
    """Parse rvlab.tdscf.xyz → atom coords and grid coords (atomic units)."""
    atoms, grid, mode = [], [], None
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line == '[Atoms] (AU)':  mode = 'atoms'; continue
            if line == '[Grid] (AU)':   mode = 'grid';  continue
            parts = line.split()
            if not parts: continue
            if mode == 'atoms':
                atoms.append([float(parts[3]), float(parts[4]), float(parts[5])])
            elif mode == 'grid':
                grid.append([float(parts[1]), float(parts[2]), float(parts[3])])
    return np.array(atoms, dtype=np.float32), np.array(grid, dtype=np.float32)

xyz_file = DATA_DIR / 'rvlab.tdscf.xyz'
atom_coords, grid_coords = parse_xyz(xyz_file)
print(f'Atoms : {len(atom_coords)}')
print(f'Grid  : {len(grid_coords)} points')
print(f'Atom coordinates (AU):\n{atom_coords}')

In [ ]:
# ── 4. THE CORE DISTANCE MACHINERY ───────────────────────────────────────────
#
# Everything from here is distance-first. No KNN by value. No index tricks.
# Physical 3D Euclidean distance drives every graph decision.

# ── 4a. Radial Basis Function (RBF) encoder ───────────────────────────────────
# Instead of passing raw distance d (a single number), we expand it into
# a vector of Gaussian bumps centered at evenly-spaced distances.
#
#   RBF_k(d) = exp( -(d - μ_k)² / (2σ²) )
#
# This lets the model learn: "interactions at distance 0.5 AU matter differently
# from interactions at 2.0 AU" — without us hard-coding which distances matter.
# This is how SchNet (Schütt et al. 2017) encodes interatomic distances.

class RBFExpansion(nn.Module):
    """Expand scalar distances into a vector of Gaussian basis functions."""
    def __init__(self, d_min=0.0, d_max=8.0, num_rbf=16):
        super().__init__()
        # Fixed (non-learned) centers spread evenly over [d_min, d_max]
        centers = torch.linspace(d_min, d_max, num_rbf)  # [num_rbf]
        self.register_buffer('centers', centers)
        self.width = 2 * ((d_max - d_min) / num_rbf) ** 2  # σ² term

    def forward(self, d):  # d: [...] scalar distances
        # Expand last dim, compute Gaussians
        d = d.unsqueeze(-1)  # [..., 1]
        return torch.exp(-((d - self.centers) ** 2) / self.width)  # [..., num_rbf]


# ── 4b. Voronoi: assign every grid point to its nearest atom ─────────────────
def assign_grid_to_atoms(atom_coords, grid_coords):
    """
    Voronoi partitioning in real 3D space.
    Returns:
        assignment  : [N_grid] int  — which atom each point belongs to
        dist_to_atom: [N_grid] float — Euclidean distance to that atom (AU)
        all_dists   : [N_grid, N_atoms] — distances to ALL atoms
    """
    all_dists  = cdist(grid_coords, atom_coords)          # [N_grid, N_atoms]
    assignment = np.argmin(all_dists, axis=1)             # [N_grid]
    dist_to_atom = all_dists[np.arange(len(grid_coords)), assignment]  # [N_grid]
    return assignment, dist_to_atom, all_dists

assignment, dist_to_atom, all_dists = assign_grid_to_atoms(atom_coords, grid_coords)

# Patch sizes
unique, counts = np.unique(assignment, return_counts=True)
for atom_id, count in zip(unique, counts):
    print(f'  Atom {atom_id}: {count} grid points | max dist = {all_dists[assignment==atom_id, atom_id].max():.3f} AU')

# Distance distribution
plt.figure(figsize=(10, 3))
plt.hist(dist_to_atom, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
plt.xlabel('Distance to nearest atom (AU)')
plt.ylabel('Grid point count')
plt.title('Distribution of grid-point-to-atom distances')
plt.tight_layout(); plt.show()
print(f'\nDistance stats: min={dist_to_atom.min():.3f}  mean={dist_to_atom.mean():.3f}  max={dist_to_atom.max():.3f} AU')

In [ ]:
# ── 4c. Intra-patch edges: distance-cutoff graph ──────────────────────────────
#
# PREVIOUS approach: no edges within a patch — just an MLP on each point alone.
# NEW approach: connect grid points within the same atom's patch IF they are
# within a cutoff radius of each other. Edge weight = distance (via RBF).
#
# This lets the model do LOCAL message passing inside each patch —
# a point near the nucleus can "tell" a nearby point what the density is doing.

def build_patch_edges(grid_coords, assignment, cutoff_au=1.5):
    """
    For each atom's patch, build edges between grid points within `cutoff_au`.
    Returns:
        edge_index: [2, E]  — patch-local edges (global grid indices)
        edge_dist : [E]     — Euclidean distance for each edge
    """
    edge_src, edge_dst, edge_dist = [], [], []
    num_atoms = assignment.max() + 1

    for atom_id in range(num_atoms):
        patch_mask = np.where(assignment == atom_id)[0]  # global indices in this patch
        patch_coords = grid_coords[patch_mask]           # [P, 3]

        # Pairwise distances inside patch
        dists = cdist(patch_coords, patch_coords)        # [P, P]
        src_local, dst_local = np.where((dists < cutoff_au) & (dists > 0))

        # Map local patch index back to global grid index
        edge_src.extend(patch_mask[src_local].tolist())
        edge_dst.extend(patch_mask[dst_local].tolist())
        edge_dist.extend(dists[src_local, dst_local].tolist())

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    edge_dist  = torch.tensor(edge_dist, dtype=torch.float32)
    return edge_index, edge_dist

# Build once — geometry is fixed across all timesteps
CUTOFF_AU = 1.5  # ~0.8 Angstrom — captures nearest neighbours in real space
patch_edge_index, patch_edge_dist = build_patch_edges(grid_coords, assignment, cutoff_au=CUTOFF_AU)
print(f'Intra-patch edges  : {patch_edge_index.shape[1]:,}')
print(f'Avg edges per node : {patch_edge_index.shape[1] / len(grid_coords):.1f}')
print(f'Edge distance range: {patch_edge_dist.min():.3f} – {patch_edge_dist.max():.3f} AU')

In [ ]:
# ── 4d. Inter-atom edges with distance as edge feature ────────────────────────
#
# Atoms are fully connected (only 4 nodes for ammonia).
# But NOW the edge carries the actual physical inter-atomic distance.
# GATv2 uses this as edge_attr — so attention between N and H is informed
# by how far apart they are. This is the direct answer to your professor's point.

def build_atom_edges(atom_coords):
    """
    Fully-connected inter-atom graph with real distances as edge features.
    Returns:
        atom_edge_index: [2, N_atoms*(N_atoms-1)]
        atom_edge_dist : [N_atoms*(N_atoms-1)]  — actual 3D distance in AU
    """
    n = len(atom_coords)
    src, dst = [], []
    for i in range(n):
        for j in range(n):
            if i != j:
                src.append(i); dst.append(j)

    src_arr = np.array(src); dst_arr = np.array(dst)
    dists = np.linalg.norm(atom_coords[src_arr] - atom_coords[dst_arr], axis=1)

    atom_edge_index = torch.tensor([src, dst], dtype=torch.long)
    atom_edge_dist  = torch.tensor(dists, dtype=torch.float32)
    return atom_edge_index, atom_edge_dist

atom_edge_index, atom_edge_dist = build_atom_edges(atom_coords)
print('Inter-atom distances (AU):')
for k in range(atom_edge_index.shape[1]):
    i, j = atom_edge_index[0, k].item(), atom_edge_index[1, k].item()
    print(f'  Atom {i} → Atom {j} : {atom_edge_dist[k]:.4f} AU')

In [ ]:
# ── 5. Global Normalization (computed once over training files) ────────────────
#
# Per-snapshot norm (old) causes scale drift across time.
# Global norm (new) gives the model a consistent reference frame.

all_files = sorted(glob.glob(str(DATA_DIR / 'rvlab.tdscf.rho.*')))
print(f'Total timestep files: {len(all_files)}')

train_cutoff = int(0.8 * len(all_files))
train_files  = all_files[:train_cutoff]
val_files    = all_files[train_cutoff:]

print('Computing global stats from training files...')
sample_vals = []
for f in train_files[::5]:  # sample every 5th file for speed
    sample_vals.append(np.loadtxt(f, usecols=1))
all_vals = np.concatenate(sample_vals)

DENSITY_MEAN = float(all_vals.mean())
DENSITY_STD  = float(all_vals.std() + 1e-8)
print(f'Global density mean : {DENSITY_MEAN:.6f}')
print(f'Global density std  : {DENSITY_STD:.6f}')

In [ ]:
# ── 6. Dataset ────────────────────────────────────────────────────────────────

class DistanceHierarchicalDataset(torch.utils.data.Dataset):
    """
    Each sample:
      grid_x       : [N_grid, 1]    normalized density at time t
      dist_to_atom : [N_grid]       distance from each grid point to its atom
      assignment   : [N_grid]       which atom each grid point belongs to
      patch_ei     : [2, E_patch]   intra-patch distance-cutoff edges
      patch_ed     : [E_patch]      distances for intra-patch edges
      atom_ei      : [2, E_atom]    inter-atom edges
      atom_ed      : [E_atom]       inter-atom distances
      y            : [N_grid, 3]    target densities at t+5, t+10, t+15
    """
    def __init__(self, files, future_offsets=(5, 10, 15)):
        self.files   = files
        self.offsets = list(future_offsets)
        self.valid_len = len(files) - max(future_offsets)

        # Pre-computed geometry (shared across all timesteps)
        self.assignment   = torch.tensor(assignment,    dtype=torch.long)
        self.dist_to_atom = torch.tensor(dist_to_atom,  dtype=torch.float32)
        self.patch_ei     = patch_edge_index
        self.patch_ed     = patch_edge_dist
        self.atom_ei      = atom_edge_index
        self.atom_ed      = atom_edge_dist

    def __len__(self):
        return self.valid_len

    def __getitem__(self, idx):
        # Current density — globally normalized
        x_raw  = np.loadtxt(self.files[idx], usecols=1).astype(np.float32)
        x_norm = (x_raw - DENSITY_MEAN) / DENSITY_STD

        # Target densities — same global normalization
        targets = []
        for off in self.offsets:
            t_raw = np.loadtxt(self.files[idx + off], usecols=1).astype(np.float32)
            targets.append((t_raw - DENSITY_MEAN) / DENSITY_STD)

        return {
            'grid_x'       : torch.tensor(x_norm, dtype=torch.float32).unsqueeze(-1),
            'dist_to_atom' : self.dist_to_atom,
            'assignment'   : self.assignment,
            'patch_ei'     : self.patch_ei,
            'patch_ed'     : self.patch_ed,
            'atom_ei'      : self.atom_ei,
            'atom_ed'      : self.atom_ed,
            'y'            : torch.tensor(np.column_stack(targets), dtype=torch.float32),
        }

train_dataset = DistanceHierarchicalDataset(train_files)
val_dataset   = DistanceHierarchicalDataset(val_files)
print(f'Train samples : {len(train_dataset)}')
print(f'Val samples   : {len(val_dataset)}')

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=1, shuffle=False)

In [ ]:
# ── 7. Distance-Aware Hierarchical GNN ───────────────────────────────────────
#
# Four phases — now every single one is guided by physical distance:
#
#  Phase 1 — Intra-patch GNN  : local message passing inside each atom's patch
#                               using distance-cutoff edges + RBF edge features
#  Phase 2 — Dist-weighted pool: aggregate patch → atom, weight by exp(-d/σ)
#                               so near-nucleus points contribute more
#  Phase 3 — Inter-atom GAT   : GATv2 between atoms, RBF(inter-atomic dist)
#                               as explicit edge_attr
#  Phase 4 — Decode to grid   : broadcast atom embedding + RBF(d_to_atom)
#                               so decoder knows how far each point is from its atom

class DistanceAwareHierarchicalGNN(nn.Module):
    def __init__(self, hidden_dim=64, num_rbf=16, num_futures=3, pool_sigma=1.0):
        super().__init__()
        self.pool_sigma = pool_sigma
        self.num_rbf    = num_rbf

        # Shared RBF expanders
        self.rbf_patch = RBFExpansion(d_min=0.0, d_max=CUTOFF_AU, num_rbf=num_rbf)
        self.rbf_atom  = RBFExpansion(d_min=0.0, d_max=10.0,      num_rbf=num_rbf)
        self.rbf_grid  = RBFExpansion(d_min=0.0, d_max=10.0,      num_rbf=num_rbf)

        # ── Phase 1: Intra-patch encoder ────────────────────────────────────
        # Node input: density(1) + RBF(d_to_atom)(num_rbf)
        # Edge input: RBF(patch_dist)(num_rbf)
        self.patch_node_proj = nn.Linear(1 + num_rbf, hidden_dim)
        self.patch_gat = GATv2Conv(
            in_channels  = hidden_dim,
            out_channels = hidden_dim // 2,
            heads        = 2,
            edge_dim     = num_rbf,   # ← RBF-encoded intra-patch distances
            concat       = True,
            add_self_loops=True,
        )
        self.patch_norm = nn.LayerNorm(hidden_dim)

        # ── Phase 3: Inter-atom GATv2 ────────────────────────────────────────
        # edge_attr = RBF(inter-atomic distance)  ← professor's key suggestion
        self.atom_gat = GATv2Conv(
            in_channels  = hidden_dim,
            out_channels = hidden_dim // 2,
            heads        = 2,
            edge_dim     = num_rbf,   # ← RBF-encoded inter-atomic distances
            concat       = True,
            add_self_loops=False,
        )
        self.atom_norm = nn.LayerNorm(hidden_dim)

        # ── Phase 4: Decoder heads (one per future timestep) ─────────────────
        # Input: atom_embedding(hidden_dim) + RBF(d_to_atom)(num_rbf) + density(1)
        decoder_in = hidden_dim + num_rbf + 1
        self.decoders = nn.ModuleList([
            nn.Sequential(
                nn.Linear(decoder_in, hidden_dim),
                nn.ReLU(),
                nn.LayerNorm(hidden_dim),
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.ReLU(),
                nn.Linear(hidden_dim // 2, 1),
            ) for _ in range(num_futures)
        ])

    def forward(self, batch):
        grid_x      = batch['grid_x'].squeeze(0).to(device)       # [N_grid, 1]
        d_to_atom   = batch['dist_to_atom'].squeeze(0).to(device)  # [N_grid]
        assignment  = batch['assignment'].squeeze(0).to(device)    # [N_grid]
        patch_ei    = batch['patch_ei'].squeeze(0).to(device)      # [2, E_patch]
        patch_ed    = batch['patch_ed'].squeeze(0).to(device)      # [E_patch]
        atom_ei     = batch['atom_ei'].squeeze(0).to(device)       # [2, E_atom]
        atom_ed     = batch['atom_ed'].squeeze(0).to(device)       # [E_atom]

        # ── Phase 1: Intra-patch message passing ─────────────────────────────
        # Node features: density + RBF(distance to own atom)
        rbf_d_grid = self.rbf_grid(d_to_atom)                     # [N_grid, num_rbf]
        h_grid = torch.cat([grid_x, rbf_d_grid], dim=-1)          # [N_grid, 1+num_rbf]
        h_grid = F.relu(self.patch_node_proj(h_grid))              # [N_grid, hidden]

        # Edge features: RBF(intra-patch distance)
        rbf_patch_e = self.rbf_patch(patch_ed)                     # [E_patch, num_rbf]

        # GAT message passing within patches
        h_grid_mp = self.patch_gat(h_grid, patch_ei, edge_attr=rbf_patch_e)  # [N_grid, hidden]
        h_grid    = self.patch_norm(h_grid + h_grid_mp)            # residual

        # ── Phase 2: Distance-weighted pooling → atom embeddings ──────────────
        # weight = exp(-d / σ)  →  points closer to nucleus contribute MORE
        weights  = torch.exp(-d_to_atom / self.pool_sigma)         # [N_grid]
        h_w      = h_grid * weights.unsqueeze(-1)                  # [N_grid, hidden]

        # Weighted sum per atom (manual scatter_add)
        num_atoms = atom_ei.max().item() + 1
        h_atoms   = torch.zeros(num_atoms, h_grid.shape[-1], device=device)
        h_atoms.scatter_add_(0, assignment.unsqueeze(-1).expand_as(h_w), h_w)

        # Normalize by sum of weights per atom
        w_sum = torch.zeros(num_atoms, device=device)
        w_sum.scatter_add_(0, assignment, weights)
        h_atoms = h_atoms / (w_sum.unsqueeze(-1) + 1e-8)          # [N_atoms, hidden]

        # ── Phase 3: Inter-atom message passing with distance edge features ───
        rbf_atom_e = self.rbf_atom(atom_ed)                        # [E_atom, num_rbf]
        h_atoms_mp = self.atom_gat(h_atoms, atom_ei, edge_attr=rbf_atom_e)  # [N_atoms, hidden]
        h_atoms    = self.atom_norm(h_atoms + h_atoms_mp)          # residual

        # ── Phase 4: Decode back to grid ──────────────────────────────────────
        # Each grid point gets its atom's updated embedding
        h_broadcast = h_atoms[assignment]                          # [N_grid, hidden]

        # Also encode how far this grid point is from its atom (via RBF)
        rbf_d_decode = self.rbf_grid(d_to_atom)                   # [N_grid, num_rbf]

        # Final context: global atom info + distance awareness + local density
        context = torch.cat([h_broadcast, rbf_d_decode, grid_x], dim=-1)  # [N_grid, hidden+rbf+1]

        # One prediction per future timestep
        preds = [dec(context).squeeze(-1) for dec in self.decoders]
        return torch.stack(preds, dim=-1)  # [N_grid, 3]


model = DistanceAwareHierarchicalGNN(
    hidden_dim  = 64,
    num_rbf     = 16,
    num_futures = 3,
    pool_sigma  = 1.0,   # AU — controls how fast pooling weight decays with distance
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')
print(model)

In [ ]:
# ── 8. Training Loop ─────────────────────────────────────────────────────────

from pathlib import Path

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10, verbose=True
)
criterion = nn.MSELoss()

save_dir = Path('../models')
save_dir.mkdir(exist_ok=True)
best_model_path = save_dir / 'best_distance_hierarchical.pt'

NUM_EPOCHS    = 100
best_val_loss = float('inf')
train_losses, val_losses = [], []
per_step_losses = {0: [], 1: [], 2: []}  # per future timestep

print(f'Training for {NUM_EPOCHS} epochs...')
print('=' * 60)

for epoch in range(NUM_EPOCHS):

    # ── Train ──
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        pred   = model(batch)                         # [N_grid, 3]
        target = batch['y'].squeeze(0).to(device)     # [N_grid, 3]
        loss   = criterion(pred, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    avg_train = epoch_loss / len(train_loader)
    train_losses.append(avg_train)

    # ── Validate ──
    model.eval()
    val_loss = 0.0
    ts_losses = [0.0, 0.0, 0.0]
    with torch.no_grad():
        for batch in val_loader:
            pred   = model(batch)
            target = batch['y'].squeeze(0).to(device)
            val_loss += criterion(pred, target).item()
            for i in range(3):
                ts_losses[i] += criterion(pred[:, i], target[:, i]).item()

    avg_val = val_loss / len(val_loader)
    val_losses.append(avg_val)
    for i in range(3):
        per_step_losses[i].append(ts_losses[i] / len(val_loader))

    scheduler.step(avg_val)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), best_model_path)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{NUM_EPOCHS} | '
              f'Train {avg_train:.5f} | '
              f'Val {avg_val:.5f} | '
              f'[t+5:{per_step_losses[0][-1]:.5f}  '
              f't+10:{per_step_losses[1][-1]:.5f}  '
              f't+15:{per_step_losses[2][-1]:.5f}]')

print(f'\nBest val loss: {best_val_loss:.6f}  (saved to {best_model_path})')

In [ ]:
# ── 9. Training Curves ────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(train_losses, label='Train', linewidth=2)
axes[0].plot(val_losses,   label='Val',   linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Overall Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

colors = ['#e74c3c', '#f39c12', '#2ecc71']
labels = ['t+5', 't+10', 't+15']
for i in range(3):
    axes[1].plot(per_step_losses[i], label=labels[i], color=colors[i], linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE Loss (val)')
axes[1].set_title('Val Loss per Future Timestep')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Distance-Aware Hierarchical GNN — Training', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 10. Full Evaluation ───────────────────────────────────────────────────────

def denormalize(x):
    return x * DENSITY_STD + DENSITY_MEAN

model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

all_preds   = {i: [] for i in range(3)}
all_targets = {i: [] for i in range(3)}

with torch.no_grad():
    for batch in val_loader:
        pred   = model(batch)                      # [N_grid, 3] normalized
        target = batch['y'].squeeze(0).to(device)
        for i in range(3):
            all_preds[i].append(denormalize(pred[:, i]).cpu().numpy())
            all_targets[i].append(denormalize(target[:, i]).cpu().numpy())

for i in range(3):
    all_preds[i]   = np.concatenate(all_preds[i])
    all_targets[i] = np.concatenate(all_targets[i])

print('=' * 65)
print('EVALUATION RESULTS (denormalized — real density units)')
print('=' * 65)
r2_list, mae_list = [], []
for i, label in enumerate(['t+5', 't+10', 't+15']):
    p, t = all_preds[i], all_targets[i]
    mae  = np.mean(np.abs(p - t))
    r2   = 1 - np.sum((t - p)**2) / np.sum((t - t.mean())**2)
    r2_list.append(r2); mae_list.append(mae)
    print(f'  {label}  →  MAE: {mae:.6f}   R²: {r2:.4f}')
print(f'\n  Average   →  MAE: {np.mean(mae_list):.6f}   R²: {np.mean(r2_list):.4f}')

In [ ]:
# ── 11. Baseline Comparison ───────────────────────────────────────────────────

persistence_maes = []
future_offsets = [5, 10, 15]

for off in future_offsets:
    errors = []
    for f_idx in range(len(val_files) - off):
        cur  = np.loadtxt(val_files[f_idx],       usecols=1)
        fut  = np.loadtxt(val_files[f_idx + off], usecols=1)
        errors.append(np.mean(np.abs(cur - fut)))
    persistence_maes.append(np.mean(errors))

print('\nBaseline vs Model (MAE in real units):')
print(f'{"":10s} {"Persistence":>14s} {"Our Model":>12s} {"Improvement":>12s}')
print('-' * 52)
for i, label in enumerate(['t+5', 't+10', 't+15']):
    imp = (persistence_maes[i] - mae_list[i]) / persistence_maes[i] * 100
    print(f'{label:10s} {persistence_maes[i]:14.6f} {mae_list[i]:12.6f} {imp:>11.1f}%')

In [ ]:
# ── 12. Prediction vs Actual Plots ────────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = ['#e74c3c', '#f39c12', '#2ecc71']
labels = ['t+5', 't+10', 't+15']

for i in range(3):
    p, t = all_preds[i], all_targets[i]

    # Scatter: predicted vs actual
    axes[0, i].scatter(t, p, alpha=0.2, s=1, color=colors[i])
    lims = [min(t.min(), p.min()), max(t.max(), p.max())]
    axes[0, i].plot(lims, lims, 'k--', linewidth=1.5, label='Perfect')
    axes[0, i].set_title(f'{labels[i]}  R²={r2_list[i]:.4f}', fontsize=12, fontweight='bold')
    axes[0, i].set_xlabel('Actual density'); axes[0, i].set_ylabel('Predicted density')
    axes[0, i].legend(); axes[0, i].grid(True, alpha=0.3)

    # Error histogram
    errors = p - t
    axes[1, i].hist(errors, bins=80, color=colors[i], alpha=0.7, edgecolor='none')
    axes[1, i].axvline(0, color='black', linewidth=1.5)
    axes[1, i].set_title(f'Error distribution — {labels[i]}')
    axes[1, i].set_xlabel('Prediction error'); axes[1, i].set_ylabel('Count')
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle('Distance-Aware Hierarchical GNN — Prediction Quality', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 13. Attention Weight Analysis (Distance vs Attention) ─────────────────────
#
# Since we feed RBF(distance) as edge_attr to GATv2, we can ask:
# does attention correlate with distance? It should — nearby atoms should
# attend more strongly.

model.eval()
sample = val_dataset[0]

# Manually run Phase 3 to extract attention weights
with torch.no_grad():
    # Re-run forward to get atom embeddings at Phase 3
    grid_x    = sample['grid_x'].to(device)
    d_to_atom = sample['dist_to_atom'].to(device)
    assign    = sample['assignment'].to(device)
    patch_ei  = sample['patch_ei'].to(device)
    patch_ed  = sample['patch_ed'].to(device)
    atom_ei   = sample['atom_ei'].to(device)
    atom_ed   = sample['atom_ed'].to(device)

    rbf_d     = model.rbf_grid(d_to_atom)
    h         = F.relu(model.patch_node_proj(torch.cat([grid_x, rbf_d], dim=-1)))
    rbf_pe    = model.rbf_patch(patch_ed)
    h         = model.patch_norm(h + model.patch_gat(h, patch_ei, edge_attr=rbf_pe))

    weights   = torch.exp(-d_to_atom / model.pool_sigma)
    h_w       = h * weights.unsqueeze(-1)
    n_atoms   = atom_ei.max().item() + 1
    h_atoms   = torch.zeros(n_atoms, h.shape[-1], device=device)
    h_atoms.scatter_add_(0, assign.unsqueeze(-1).expand_as(h_w), h_w)
    w_sum     = torch.zeros(n_atoms, device=device)
    w_sum.scatter_add_(0, assign, weights)
    h_atoms   = h_atoms / (w_sum.unsqueeze(-1) + 1e-8)

    rbf_ae    = model.rbf_atom(atom_ed)
    _, (attn_idx, attn_w) = model.atom_gat(
        h_atoms, atom_ei, edge_attr=rbf_ae, return_attention_weights=True
    )

attn_mean = attn_w.mean(dim=-1).cpu().numpy()
inter_dist = atom_ed.cpu().numpy()

plt.figure(figsize=(7, 5))
plt.scatter(inter_dist, attn_mean, s=80, c='steelblue', edgecolors='k', linewidths=0.5)
for k in range(len(inter_dist)):
    src, dst = attn_idx[0, k].item(), attn_idx[1, k].item()
    plt.annotate(f'{src}→{dst}', (inter_dist[k], attn_mean[k]), fontsize=8, ha='left')
plt.xlabel('Inter-atomic distance (AU)', fontsize=12)
plt.ylabel('Mean attention weight', fontsize=12)
plt.title('Does the model attend more to closer atoms?\n(Distance-driven attention)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Summary — What Distance-Awareness Changed

| Design Choice | Physical Justification |
|---|---|
| **RBF encoding of distances** | Model can learn which distance ranges matter (bond length vs. long-range) |
| **Distance-cutoff intra-patch edges** | Only physically nearby grid points exchange messages — no spurious long-range connections |
| **Distance-weighted pooling** (`exp(-d/σ)`) | Near-nucleus points dominate — aligns with how density actually behaves |
| **RBF(inter-atomic dist) as GAT edge_attr** | Atom-atom attention is informed by the actual bond geometry — N–H vs H–H are different distances |
| **RBF(d_to_atom) in decoder** | Decoder knows how far each grid point is from its atom — physically correct decay |
| **Global normalization** | Consistent scale across all timesteps — no scale drift during dynamics |

The key insight your professor pointed to: **distance is not just a feature — it is the organizing principle of all quantum chemistry.** Every part of the architecture now reflects that.